# External File Storage

## What you'll learn

- When to store files externally instead of embedding them in Delta Lake
- The two external-content patterns: one-to-one (large files) and many-to-one (record bundles)
- How Artisan manages external files in `files_root`

**Prerequisites:** [First Pipeline](../getting-started/01-first-pipeline.ipynb),
[Storage Layout](05-storage-and-logging.ipynb)  
**Estimated time:** 10 minutes  
**GPU required:** No.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from artisan.operations.examples import LargeFileGenerator, RecordBundleGenerator
from artisan.orchestration import Backend, PipelineManager
from artisan.storage.core.artifact_store import ArtifactStore
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

In [ ]:
env = tutorial_setup("external_files")
DELTA_ROOT = env.delta_root

pipeline = PipelineManager.create(
    name="external_files",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# files_root defaults to delta_root.parent / "files"
files_root = DELTA_ROOT.parent / "files"
print(f"files_root: {files_root}")

## When to use external storage

Most artifact types embed their content directly in Delta Lake as a
binary column. `DataArtifact` stores raw CSV bytes, `MetricArtifact`
stores JSON — all inside Parquet files. This works well for data up to
a few megabytes per artifact.

External storage is for two scenarios where embedding breaks down:

| Scenario | Example | Pattern |
|----------|---------|--------|
| **Large files** | Model weights, embeddings, HDF5 datasets | One artifact = one file |
| **Multi-record files** | JSONL datasets, FASTA sequences, SDF molecules | Many artifacts share one file |

With external storage, Delta tracks only metadata (hash, size, name).
The actual file content lives in `files_root`, a directory managed by
Artisan alongside `delta_root`.

## Large files

`LargeFileArtifact` represents a single file too large for a Parquet
binary column. Each artifact maps to exactly one file in `files_root`.

In [ ]:
pipeline.run(
    operation=LargeFileGenerator,
    name="generate_files",
    params={"count": 3, "file_size_bytes": 10_000, "seed": 42},
    backend=Backend.LOCAL,
)

In [ ]:
inspect_pipeline(DELTA_ROOT)

In [ ]:
inspect_step(DELTA_ROOT, step_number=0)

### The files_root directory

External files are organized under `files_root` by step number and
worker. Each worker writes to its own subdirectory to avoid collisions.

In [ ]:
def show_tree(path: Path, prefix: str = "", max_depth: int = 3, depth: int = 0) -> None:
    """Display a directory tree up to max_depth."""
    if depth >= max_depth or not path.is_dir():
        return
    entries = sorted(path.iterdir())
    for i, entry in enumerate(entries):
        connector = (
            "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        )
        suffix = "/" if entry.is_dir() else ""
        print(f"{prefix}{connector}{entry.name}{suffix}")
        if entry.is_dir():
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            show_tree(entry, prefix + extension, max_depth, depth + 1)


print(f"{files_root.name}/")
show_tree(files_root, max_depth=4)

### Inspecting artifact metadata

Each `LargeFileArtifact` stores the file's hash, size, and path in
Delta. The actual bytes live at `external_path`.

In [ ]:
store = ArtifactStore(DELTA_ROOT)
large_ids = sorted(store.load_artifact_ids_by_type("large_file"))
large_artifacts = store.get_artifacts_by_type(large_ids, "large_file")

for art_id, art in sorted(large_artifacts.items()):
    print(
        f"{art.original_name}{art.extension}  size={art.size_bytes:,}  path=.../{Path(art.external_path).name}"
    )

## Record bundles

`RecordBundleArtifact` represents one record within a shared JSONL
file. Multiple artifacts point to the same `external_path`, each
identified by a `record_id`. This is the pattern for any file
format where many independently addressable items live in one file
(JSONL, FASTA, SDF).

In [ ]:
pipeline.run(
    operation=RecordBundleGenerator,
    name="generate_records",
    params={"count": 5, "fields_per_record": 3, "seed": 42},
    backend=Backend.LOCAL,
)

In [ ]:
inspect_step(DELTA_ROOT, step_number=1)

### Shared external path

All five artifacts reference the same JSONL file. Each tracks its own
`record_id`, `content_hash`, and `size_bytes`.

In [ ]:
bundle_ids = sorted(store.load_artifact_ids_by_type("record_bundle"))
bundle_artifacts = store.get_artifacts_by_type(bundle_ids, "record_bundle")

paths = set()
for art in bundle_artifacts.values():
    paths.add(art.external_path)
    print(f"  record_id={art.record_id}  size={art.size_bytes}")

print(f"\nAll {len(bundle_artifacts)} artifacts share {len(paths)} file")

### Reading the JSONL file

The JSONL file contains one JSON object per line. Each line has a
`record_id` and a `values` dict with the generated data.

In [ ]:
jsonl_path = next(iter(paths))
with open(jsonl_path) as f:
    for i, line in enumerate(f):
        if i >= 2:
            print(f"  ... ({len(bundle_artifacts) - 2} more records)")
            break
        record = json.loads(line)
        print(f"  {record['record_id']}: {record['values']}")

In [ ]:
result = pipeline.finalize()
print(f"Pipeline complete: {result['total_steps']} steps")

## Comparison

| | Embedded (DataArtifact) | One-to-one (LargeFileArtifact) | Many-to-one (RecordBundleArtifact) |
|---|---|---|---|
| **Content stored in** | Delta Lake (Parquet binary column) | `files_root` | `files_root` |
| **Artifacts per file** | 1 | 1 | Many |
| **Good for** | CSVs, configs, metrics | Model weights, embeddings, HDF5 | JSONL, FASTA, SDF |
| **Default consumption** | Materialize to disk | Read via `external_path` | Read via `external_path` |
| **Consolidation needed** | No | No | Yes (per-worker files) |

## Summary

- **`LargeFileArtifact`** stores one large file per artifact in
  `files_root`. Delta tracks hash, size, and path.
- **`RecordBundleArtifact`** stores many records in a shared JSONL
  file. Each artifact has a `record_id` addressing one record.
- External files live at `files_root/{step}/workers/{run_id}/`.
- Use `ArtifactStore` to hydrate artifacts and access `external_path`.

## Next steps

- [Post-Step Consolidation](12-post-step-consolidation.ipynb) —
  Merge per-worker record bundles into a single file
- [Storage Layout](05-storage-and-logging.ipynb) —
  The three-path model and Delta Lake table structure
- [SLURM Execution](07-slurm-execution.ipynb) —
  Run steps on HPC clusters where external storage becomes essential